# Notebook 01 — Funciones agregadas

Primer sub-bloque del Tema 05. Las **funciones agregadas** (también llamadas *de agregación*) toman muchas filas y devuelven **una sola**. Son la base de cualquier query analítica: contar pedidos, sumar ventas, promediar precios, encontrar el máximo y el mínimo.

En este notebook cubres `count`, `sum`, `avg`, `min`, `max`, `string_agg`, `array_agg`, la **trampa de los NULLs** que afecta a todas ellas, y la diferencia crítica entre **`WHERE`** (filtra antes de agregar) y **`HAVING`** (filtra después).

Todo el SQL se ejecuta directamente desde el notebook con **JupySQL** — el resultado se muestra como tabla nativa, sin tener que envolver cada query en `pd.read_sql`.

**Contenido de este notebook:**

- [Setup — conexión con JupySQL](#setup--conexión-con-jupysql)
- [Las 5 agregadas básicas — `count`, `sum`, `avg`, `min`, `max`](#las-5-agregadas-básicas--count-sum-avg-min-max)
- [`COUNT(*)` vs `COUNT(col)` vs `COUNT(DISTINCT col)` — la diferencia que sí importa](#count-vs-countcol-vs-countdistinct-col--la-diferencia-que-sí-importa)
- [La trampa de los NULLs — todas las agregadas los ignoran (silenciosamente)](#la-trampa-de-los-nulls--todas-las-agregadas-los-ignoran-silenciosamente)
- [`GROUP BY` — repaso y el patrón estándar](#group-by--repaso-y-el-patrón-estándar)
- [`WHERE` vs `HAVING` — antes y después del agregado](#where-vs-having--antes-y-después-del-agregado)
- [Agregadas de concatenación — `string_agg` y `array_agg`](#agregadas-de-concatenación--string_agg-y-array_agg)
- [Agregadas sobre datos sucios — caso Airbnb](#agregadas-sobre-datos-sucios--caso-airbnb)

## Setup — conexión con JupySQL

**JupySQL** es la extensión moderna de Jupyter para SQL. Permite escribir queries directamente en celdas con el *magic* `%%sql` y muestra los resultados como tabla.

Si es la primera vez, descomenta la línea de `%pip install` y ejecútala una sola vez. Después, en cada sesión: cargar la extensión y conectar el engine.

**Dos formas de invocarlo:**

- **`%sql`** — un solo query en una sola línea (útil para conectar el engine y para smoke tests).
- **`%%sql`** — un query multilínea que ocupa toda la celda. Debe ir como primera línea de la celda.

In [ ]:
# Solo la primera vez — descomenta, corre, reinicia el kernel (Kernel → Restart).
# %pip install jupysql sqlalchemy psycopg2-binary --quiet

%load_ext sql

In [ ]:
from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

# Registrar el engine como conexión default para %sql / %%sql
%sql engine

# Smoke test
%sql SELECT current_database() AS db, current_user AS usuario;

## Las 5 agregadas básicas — `count`, `sum`, `avg`, `min`, `max`

El SQL estándar define cinco funciones de agregación que todo motor implementa:

| Función | Qué hace | Tipo de columna |
|---|---|---|
| `COUNT(...)` | Cuenta filas | Cualquiera |
| `SUM(col)` | Suma valores | Numéricos |
| `AVG(col)` | Promedio | Numéricos |
| `MIN(col)` | Mínimo | Numéricos, fechas, strings (orden alfabético) |
| `MAX(col)` | Máximo | Igual que `MIN` |

Aplicadas sobre `fact_sales` completo, sin `GROUP BY`, te dan **un resumen global** de toda la tabla:

In [ ]:
%%sql
SELECT
    COUNT(*)                          AS total_lineas,
    SUM(quantity)                     AS total_unidades,
    ROUND(AVG(quantity)::NUMERIC, 2)  AS promedio_unidades,
    MIN(quantity)                     AS min_unidades,
    MAX(quantity)                     AS max_unidades,
    SUM(line_total)                   AS ventas_netas_totales
FROM northwind_dwh.fact_sales;

Sin `GROUP BY`, **toda la tabla es un solo grupo** — la query devuelve exactamente una fila con los agregados.

El `ROUND(AVG(quantity)::NUMERIC, 2)` aplica dos cosas a la vez: el cast a `NUMERIC` (porque `AVG` sobre `SMALLINT` devuelve un `NUMERIC` con muchos decimales por default) y el redondeo a 2 dígitos. Es el patrón estándar cuando quieres un promedio presentable.

## `COUNT(*)` vs `COUNT(col)` vs `COUNT(DISTINCT col)` — la diferencia que sí importa

Los tres se ven parecidos pero **cuentan cosas distintas**. Es uno de los errores más comunes en queries de BI:

| Forma | Qué cuenta | NULL |
|---|---|---|
| `COUNT(*)` | **Todas** las filas del grupo | Las cuenta (los NULLs son filas reales) |
| `COUNT(col)` | Filas donde `col` **NO es NULL** | Las ignora |
| `COUNT(DISTINCT col)` | Valores **únicos** de `col` (no NULL) | Las ignora |

Demostración en `fact_sales`. `shipped_date_key` es `NULL` para pedidos que aún no se han enviado — el contraste se ve directo:

In [ ]:
%%sql
SELECT
    COUNT(*)                          AS total_lineas,
    COUNT(shipped_date_key)           AS lineas_enviadas,
    COUNT(DISTINCT shipped_date_key)  AS dias_distintos_envio,
    COUNT(*) - COUNT(shipped_date_key) AS lineas_pendientes
FROM northwind_dwh.fact_sales;

Interpretación:

- `total_lineas` = 2 155 — el tamaño total de la fact.
- `lineas_enviadas` = 2 082 — las que ya tienen fecha de envío (NO NULL).
- `dias_distintos_envio` ≈ 189 — cuántas fechas únicas hay (cada fecha cubre varios pedidos).
- `lineas_pendientes` = 73 — diferencia que confirma cuántos pedidos están sin enviar.

**Punto pedagógico:** si por costumbre escribes siempre `COUNT(*)` esperando obtener "el conteo", vas a contar cosas que pensabas que estaban excluidas. La distinción matters cuando reportas "clientes activos" o "pedidos completados" — `COUNT(*)` no es lo mismo que `COUNT(columna_con_NULLs)`.

## La trampa de los NULLs — todas las agregadas los ignoran (silenciosamente)

**Excepto `COUNT(*)`**, las cinco agregadas básicas **descartan los NULLs antes de calcular** — sin avisar. Suena inocuo pero produce resultados engañosos:

In [ ]:
%%sql
SELECT
    AVG(shipped_date_key) AS avg_smartkey_envio,  -- promedia solo las 2 082 con fecha
    AVG(quantity)         AS avg_quantity         -- aquí no hay NULLs, da el promedio real
FROM northwind_dwh.fact_sales;

Caso típico de bug en BI: el reporte dice *"el tiempo promedio de envío es X días"*, pero ese promedio se calculó solo sobre los pedidos ya enviados — los pendientes se descartaron silenciosamente. El número se ve plausible, pero subestima la realidad operativa.

**Reglas operativas:**

1. **Antes de un `AVG`/`SUM`/`MIN`/`MAX` sobre una columna nullable, hazte la pregunta: ¿quiero ignorar los NULLs o tratarlos como 0?**
2. Si quieres tratarlos como `0`, usa `COALESCE`: `SUM(COALESCE(col, 0))`.
3. Si quieres **excluir explícitamente** las filas con NULL, agrega un `WHERE col IS NOT NULL` — vuelves visible la decisión.
4. Si quieres contar todas las filas sin importar NULL, usa `COUNT(*)` — es la única agregada que **no** ignora NULLs (porque cuenta filas, no valores).

## `GROUP BY` — repaso y el patrón estándar

Hasta aquí las queries devolvían **una sola fila** porque agregaban toda la tabla. Para **agregar por subconjuntos** usas `GROUP BY`: pandas-estilo `df.groupby('col').agg(...)` pero en SQL.

Patrón canónico: tu `SELECT` puede tener (a) columnas agregadas, (b) columnas listadas en `GROUP BY`. **Nada más.** Listar una columna que no está en ningún `GROUP BY` ni dentro de una función agregada es un error de sintaxis en PostgreSQL.

Ejemplo: ventas netas por categoría de producto.

In [ ]:
%%sql
SELECT
    dp.category_name,
    COUNT(*)                              AS lineas,
    SUM(fs.quantity)                      AS unidades_totales,
    ROUND(SUM(fs.line_total), 2)          AS ventas_netas,
    ROUND(AVG(fs.unit_price)::NUMERIC, 2) AS precio_promedio
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_product dp USING (product_key)
GROUP BY  dp.category_name
ORDER BY  ventas_netas DESC;

Lectura: cada fila del resultado es un grupo (una categoría); los agregados se calculan **dentro** de ese grupo. *Beverages* y *Dairy Products* típicamente lideran en Northwind.

**Multi-columna en `GROUP BY`:** combinar varias columnas crea grupos más finos. Ventas por categoría y año:

In [ ]:
%%sql
SELECT
    dp.category_name,
    dd.year,
    COUNT(*)                       AS lineas,
    ROUND(SUM(fs.line_total), 2)   AS ventas_netas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_product dp USING (product_key)
JOIN      northwind_dwh.dim_date    dd ON dd.date_key = fs.order_date_key
GROUP BY  dp.category_name, dd.year
ORDER BY  dp.category_name, dd.year;

## `WHERE` vs `HAVING` — antes y después del agregado

Una confusión clásica. Las dos cláusulas filtran, pero **en momentos distintos del pipeline de la query**:

| Cláusula | Filtra | Cuándo se evalúa |
|---|---|---|
| `WHERE` | Filas individuales | **Antes** del agregado |
| `HAVING` | Grupos completos (post-agregado) | **Después** del agregado |

Regla mental: `WHERE` solo puede usar columnas **base** de las tablas. `HAVING` puede usar **expresiones agregadas** (`SUM(...)`, `COUNT(...)`).

Caso: empleados de USA con más de $100 000 USD en ventas netas. Aquí se necesitan ambos filtros — uno antes (`country = 'USA'`), uno después (`SUM > 100000`).

In [ ]:
%%sql
SELECT
    de.full_name,
    COUNT(*)                       AS lineas,
    ROUND(SUM(fs.line_total), 2)   AS ventas_netas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_employee de USING (employee_key)
WHERE     de.country = 'USA'                   -- filtra filas antes de agregar
GROUP BY  de.full_name
HAVING    SUM(fs.line_total) > 100000          -- filtra grupos después de agregar
ORDER BY  ventas_netas DESC;

**Orden de ejecución mental** de una query SQL con agregación:

```
1. FROM + JOIN        →  forma el conjunto de filas inicial
2. WHERE              →  filtra filas
3. GROUP BY           →  agrupa las filas restantes
4. agregadas (SUM…)   →  calcula los valores por grupo
5. HAVING             →  filtra grupos según los agregados
6. SELECT             →  proyecta las columnas finales
7. ORDER BY           →  ordena el resultado
8. LIMIT              →  recorta la cola
```

Este orden explica por qué `WHERE SUM(x) > 100` **no funciona** (en el paso 2 el `SUM` aún no existe) y por qué `HAVING product_id > 10` sin agregado también es sospechoso (estás filtrando un grupo por algo que es del nivel de fila — debería ir en `WHERE`).

## Agregadas de concatenación — `string_agg` y `array_agg`

Las cinco básicas devuelven números. Pero a veces quieres **listar los valores** de un grupo, no agregarlos numéricamente. Para eso PostgreSQL tiene:

- **`string_agg(col, sep)`** — concatena los valores en un solo string separado por `sep`.
- **`array_agg(col)`** — los devuelve como un `ARRAY` de PostgreSQL.

Ambas aceptan un `ORDER BY` **dentro** del paréntesis para garantizar orden estable (sin él, el orden de concatenación es indefinido).

In [ ]:
%%sql
SELECT
    de.country,
    COUNT(*) AS empleados,
    STRING_AGG(de.full_name, ', ' ORDER BY de.full_name) AS lista_empleados
FROM     northwind_dwh.dim_employee de
GROUP BY de.country
ORDER BY de.country;

In [ ]:
%%sql
SELECT
    dp.category_name,
    COUNT(*) AS productos,
    ARRAY_AGG(dp.product_name ORDER BY dp.product_name) AS lista_productos
FROM      northwind_dwh.dim_product dp
GROUP BY  dp.category_name
ORDER BY  productos DESC
LIMIT 3;

Casos típicos:

- **Reportes legibles para humanos** — "empleados de USA: Andrew Fuller, Janet Leverling, Margaret Peacock".
- **Inputs para herramientas que esperan arrays** — Power BI, Python, jq sobre JSON.
- **Detectar valores anómalos** — `string_agg` te muestra de un vistazo qué valores únicos aparecen en un grupo.

## Agregadas sobre datos sucios — caso Airbnb

En el DWH de Northwind las columnas numéricas ya son `NUMERIC`/`INT` — `SUM` y `AVG` funcionan sin sorpresas. En **Airbnb (bronze)** todo es `TEXT` — incluyendo el precio. Para agregar tienes que **castear primero**.

Mira la diferencia entre intentar agregar sobre la columna cruda vs limpiarla con `REPLACE` + `CAST`:

In [ ]:
%%sql
-- price viene como TEXT con formato "$1,234.00"
SELECT price, COUNT(*) AS cuantos
FROM    airbnb.listings
WHERE   price IS NOT NULL
GROUP BY price
ORDER BY cuantos DESC
LIMIT 5;

In [ ]:
%%sql
-- Limpiar y castear inline: precio promedio por alcaldía
SELECT
    neighbourhood_cleansed                                                AS alcaldia,
    COUNT(*)                                                              AS listings,
    ROUND(AVG(REPLACE(REPLACE(price, '$', ''), ',', '')::NUMERIC), 2)     AS precio_promedio,
    MIN(REPLACE(REPLACE(price, '$', ''), ',', '')::NUMERIC)               AS precio_minimo,
    MAX(REPLACE(REPLACE(price, '$', ''), ',', '')::NUMERIC)               AS precio_maximo
FROM     airbnb.listings
WHERE    price IS NOT NULL
GROUP BY neighbourhood_cleansed
ORDER BY listings DESC
LIMIT 5;

El `REPLACE(REPLACE(price, '$', ''), ',', '')::NUMERIC` quita el símbolo `$`, las comas de miles, y castea a `NUMERIC` antes del agregado. En el Notebook 02 verás formas más limpias de hacer este parseo con `regexp_replace` — por ahora, el patrón `REPLACE` anidado es suficiente.

**Punto pedagógico:** cuando una columna *parece* numérica pero está guardada como `TEXT`, las agregadas **fallan o mienten**. Casting explícito es parte del trabajo analítico — no es un detalle, es donde se decide si el reporte es correcto.

## Cierre

Lo que cubriste en este notebook:

| Tema | Comando clave |
|---|---|
| Las 5 agregadas básicas | `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` |
| Distinción importante | `COUNT(*)` vs `COUNT(col)` vs `COUNT(DISTINCT col)` |
| Manejo de NULLs | Todas ignoran NULL silenciosamente, excepto `COUNT(*)` |
| Agrupación | `GROUP BY` (una o múltiples columnas) |
| Filtros pre vs post agregado | `WHERE` vs `HAVING` |
| Concatenación agregada | `STRING_AGG`, `ARRAY_AGG` con `ORDER BY` interno |
| Casting para agregar datos sucios | `REPLACE(...)::NUMERIC` con Airbnb |

El siguiente notebook (**02 — Funciones de strings**) profundiza en el parseo de texto: `upper`, `trim`, `position`, `substring`, `split_part`, `regexp_replace`, `like`/`ilike`, `concat` y `format`. El caso pedagógico fuerte: limpiar y extraer información de columnas sucias de Airbnb (`bathrooms_text`, `price`, `host_response_rate`, `amenities`).

---

<p align="center">
<a href="Readme.md">← Volver al índice del Tema 05</a> | <a href="02_funciones_de_strings.ipynb">Siguiente: Notebook 02 — Funciones de strings →</a>
</p>